# 02 · Train the tokenizer and compare it with GPT-4o, Llama 3, Qwen3

Needs the corpus from notebook 01 (the `splits/` folder on Drive). **CPU runtime is fine.**
About 15-25 minutes in total.

1. Train `hinglish16k` (ours) and `english16k` (English-only baseline, same size).
2. Measure tokens-per-word and characters-per-token on English, Hindi and Roman Hinglish
   for ours + GPT-2, GPT-4o, Llama 3, Qwen3.
3. Save plots/tables to Drive and download a zip to commit into the repo.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    KEEP = "/content/drive/MyDrive/hinglish-nanolm/data"
else:
    KEEP = os.path.abspath("../data")
print(os.listdir(f"{KEEP}/splits"))

In [ ]:
REPO = "https://github.com/naqi7272/Hinglish-nanoLM.git"
BRANCH = "claude/hinglish-nanolm-setup-finaye"
if IN_COLAB:
    %cd /content
    !git clone -q -b {BRANCH} {REPO} code || (cd code && git pull -q)
    %cd /content/code
    !pip install -q -e ".[compare]"

Optional: the official Llama 3 repo is gated. Without a token we automatically use an
ungated mirror with the identical tokenizer. To use your own token, add it in Colab's
🔑 Secrets panel as `HF_TOKEN` (with notebook access on).

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception:
    print("no HF_TOKEN: using public mirrors (that's fine)")

## 1. Train the tokenizers (~5-10 min)

In [ ]:
!python scripts/train_tokenizer.py --data-dir {KEEP}

## 2. Compare (~5 min)

In [ ]:
!python scripts/compare_tokenizers.py --data-dir {KEEP}

In [ ]:
from IPython.display import Image, Markdown, display
display(Image("plots/fertility.png"))
display(Image("plots/chars_per_token.png"))
display(Markdown(open("results/segmentations.md", encoding="utf-8").read()))

## 3. Save results
Copies everything to Drive and downloads `m2b_results.zip`. On your laptop, unzip it inside
the `Hinglish-nanoLM` folder and push (see the chat for the exact commands).

In [ ]:
import shutil
os.makedirs("tokenizers", exist_ok=True)
for name in ("hinglish16k", "english16k"):
    shutil.copy(f"{KEEP}/tokenizers/{name}.json", f"tokenizers/{name}.json")
for d in ("results", "plots", "tokenizers"):
    shutil.copytree(d, f"{KEEP}/../m2b/{d}", dirs_exist_ok=True)
!zip -qr m2b_results.zip results plots tokenizers
if IN_COLAB:
    from google.colab import files
    files.download("m2b_results.zip")